# Full Dataset Image Pre-processing

Builds four preprocessing variants for an ablation: baseline resize, global contrast stretch, contrast stretch + CLAHE, and an over-processed median-blur stack.

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path

SOURCE_PATHS = {
    "SyntheticData": r"D:\Desktop\Sesame Datasets\Synthetic Data"
}
BASE_OUTPUT = r"D:\Desktop\\Sesame Datasets\Sesame_Preprocessing_Study"

def apply_contrast_stretch(img):
    """Global Linear Normalization"""
    channels = cv2.split(img)
    stretched = [cv2.normalize(c, None, 0, 255, cv2.NORM_MINMAX) for c in channels]
    return cv2.merge(stretched)

def apply_clahe(img):
    """Local Adaptive Histogram Equalization"""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_enhanced = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l_enhanced, a, b)), cv2.COLOR_LAB2BGR)

def process_image(img, trial):
    img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_CUBIC)
    
    if trial == "Trial_A":
        return img
    
    elif trial == "Trial_B":
        return apply_contrast_stretch(img)
    
    elif trial == "Trial_C":
        img = apply_contrast_stretch(img)
        return apply_clahe(img)
    
    elif trial == "Trial_D":
        img = cv2.medianBlur(img, 3)
        img = apply_contrast_stretch(img)
        return apply_clahe(img)
    
    return img

trials = ["Trial_A", "Trial_B", "Trial_C", "Trial_D"]

for trial in trials:
    print(f"\nBuilding {trial}...")
    
    for src_name, src_path in SOURCE_PATHS.items():
        if not os.path.exists(src_path): continue
        
        for root, dirs, files in os.walk(src_path):
            class_name = os.path.basename(root)
            if not files: continue
            
            dest_dir = Path(BASE_OUTPUT) / trial / f"{src_name}_Clean" / class_name
            os.makedirs(dest_dir, exist_ok=True)
            
            for file in tqdm(files, desc=f"  Processing {src_name} - {class_name}", leave=False):
                if not file.lower().endswith(('.jpg', '.png', '.jpeg')): continue
                
                img = cv2.imread(os.path.join(root, file))
                if img is None: continue
                
                processed_img = process_image(img, trial)
                
                cv2.imwrite(str(dest_dir / f"{trial}_{file}"), processed_img)

print(f"\nABLATION FACTORY COMPLETE! All 4 trials stored in: {BASE_OUTPUT}")